# EE508 — Feature Extraction via Local Tunnel

**How this works:**
1. Your PC runs a local HTTP server serving the `Charades_v1_480/` directory
2. `cloudflared` creates a public HTTPS tunnel to that server  
3. This Colab notebook streams each video over the tunnel, extracts features on GPU, saves to Drive
4. Raw videos never leave your hard disk — only `features/` (~2.5 GB) goes to Drive

**Before running this notebook:**
```bash
# In a terminal on your PC:
cd ~/studies/project/ee508
bash scripts/serve_videos.sh
```
Copy the `https://xxxx.trycloudflare.com` URL into `TUNNEL_URL` below.

In [1]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('CUDA:', torch.cuda.is_available())

GPU: Tesla T4
CUDA: True


## Step 1 — Mount Drive & Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/ee508'
FEATURES_DIR = f'{DRIVE_ROOT}/features'
os.makedirs(FEATURES_DIR, exist_ok=True)
print('Drive mounted. Features will be saved to:', FEATURES_DIR)

Mounted at /content/drive
Drive mounted. Features will be saved to: /content/drive/MyDrive/ee508/features


## Step 2 — Upload Code & Set Tunnel URL

In [4]:
# ── Paste your tunnel URL here ──────────────────────────────────────────────
TUNNEL_URL = 'https://spine-they-william-reporting.trycloudflare.com'   # <-- CHANGE THIS
# ────────────────────────────────────────────────────────────────────────────

import requests
r = requests.get(TUNNEL_URL, timeout=10)
print('Tunnel reachable:', r.status_code == 200 or r.status_code == 404)
print('Status:', r.status_code)

Tunnel reachable: True
Status: 200


In [6]:
import os, sys

# Fix: the code is at MyDrive/ee508 directly, not MyDrive/ee508/code
DRIVE_ROOT = '/content/drive/MyDrive/ee508'

# Remove stale symlink if it exists
if os.path.islink('/content/ee508'):
    os.unlink('/content/ee508')

# Symlink the drive root (which IS the code) to /content/ee508
os.symlink(DRIVE_ROOT, '/content/ee508')
sys.path.insert(0, '/content/ee508')
os.chdir('/content/ee508')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Working directory: /content/drive/MyDrive/ee508
Files: ['requirements.txt', 'train.py', 'demo.py', 'data', 'src', 'scripts', 'tests', 'features', 'EE508_Colab.ipynb']


In [7]:
!pip install -q transformers tqdm scikit-learn opencv-python-headless

## Step 3 — Get Video List from Annotations

In [8]:
import json

# frame_indices.json was generated locally — upload it to Drive/ee508/data/
FRAME_IDX_PATH = f'{DRIVE_ROOT}/data/frame_indices.json'

with open(FRAME_IDX_PATH) as f:
    frame_indices = json.load(f)

video_ids = list(frame_indices.keys())
print(f'Total videos to process: {len(video_ids)}')

# Skip already done
done = set(os.listdir(FEATURES_DIR))
video_ids = [v for v in video_ids if v not in done]
print(f'Remaining: {len(video_ids)}')

Total videos to process: 3728
Remaining: 1614


## Step 4 — Load Models

In [9]:
import torch
from transformers import CLIPModel, CLIPProcessor
from transformers import DetrForObjectDetection, DetrImageProcessor

device = torch.device('cuda')

print('Loading CLIP...')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch16')
for p in clip_model.parameters(): p.requires_grad_(False)

print('Loading DETR...')
detr_model = DetrForObjectDetection.from_pretrained('facebook/detr-resnet-50').to(device).eval()
detr_proc  = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
for p in detr_model.parameters(): p.requires_grad_(False)

print('Models ready on', device)

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

Models ready on cuda


## Step 5 — Extract Features (streams video from your PC)

In [10]:
import requests, tempfile, os, torch, numpy as np
import cv2
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ── Constants ────────────────────────────────────────────────────────────────
K = 5
CLASS_EMBED_DIM = 64
COCO_TO_LOCAL = {'person':0,'chair':1,'dining table':2,'cup':3,
                 'cell phone':4,'book':5,'laptop':6}

def download_video(tunnel_url, vid):
    url = f'{tunnel_url}/{vid}.mp4'
    r = requests.get(url, stream=True, timeout=30)
    if r.status_code != 200:
        return None
    tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
    for chunk in r.iter_content(chunk_size=1 << 20):
        tmp.write(chunk)
    tmp.flush()
    return tmp.name

def read_frames(video_path, indices):
    cap = cv2.VideoCapture(video_path)
    frames_bgr, frames_pil = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, fr = cap.read()
        if not ret:
            fr = frames_bgr[-1].copy() if frames_bgr else np.zeros((224,224,3),np.uint8)
        frames_bgr.append(fr)
        frames_pil.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames_bgr, frames_pil

def extract_clip(frames_pil):
    tokens = []
    for i in range(0, len(frames_pil), 8):
        batch = frames_pil[i:i+8]
        inp = clip_proc(images=batch, return_tensors='pt').pixel_values.to(device)
        with torch.no_grad():
            out = clip_model.vision_model(pixel_values=inp)
        tokens.append(out.pooler_output.cpu())
    return torch.cat(tokens)  # [32, 512]

def extract_framediff(frames_bgr):
    feats = [torch.zeros(49)]
    for t in range(1, len(frames_bgr)):
        g1 = cv2.cvtColor(frames_bgr[t], cv2.COLOR_BGR2GRAY).astype(np.float32)/255
        g0 = cv2.cvtColor(frames_bgr[t-1], cv2.COLOR_BGR2GRAY).astype(np.float32)/255
        diff = torch.tensor(np.abs(g1 - g0)).unsqueeze(0).unsqueeze(0)
        feats.append(F.adaptive_avg_pool2d(diff, (7,7)).flatten())
    return torch.stack(feats)  # [32, 49]

def extract_objects(frames_pil):
    id2label = detr_model.config.id2label
    all_tokens = []
    for pil in frames_pil:
        inp = detr_proc(images=pil, return_tensors='pt').pixel_values.to(device)
        with torch.no_grad():
            out = detr_model(inp, output_hidden_states=True)
        logits = out.logits[0]; bboxes = out.pred_boxes[0]
        detr_feats = out.decoder_hidden_states[-1][0] if out.decoder_hidden_states else torch.zeros(logits.shape[0], 256)
        probs = logits.softmax(-1)[:,:-1]
        scores, cids = probs.max(-1)
        topk = scores.topk(min(K, scores.shape[0])).indices
        tokens = []
        for idx in topk:
            cname = id2label.get(cids[idx].item(),'')
            local = COCO_TO_LOCAL.get(cname, -1)
            ce = torch.zeros(CLASS_EMBED_DIM)
            if local >= 0: ce[local] = 1.0
            tokens.append(torch.cat([ce, bboxes[idx].cpu(), detr_feats[idx].cpu()[:256]]))
        while len(tokens) < K:
            tokens.append(torch.zeros(324))
        all_tokens.append(torch.stack(tokens[:K]))
    return torch.stack(all_tokens)  # [32, 5, 324]

# ── Main extraction loop ─────────────────────────────────────────────────────
errors = []
for vid in tqdm(video_ids):
    out_dir = os.path.join(FEATURES_DIR, vid)
    if os.path.exists(os.path.join(out_dir, 'clip.pt')):
        continue

    tmp_path = download_video(TUNNEL_URL, vid)
    if tmp_path is None:
        errors.append(vid)
        continue

    try:
        indices = frame_indices[vid]
        frames_bgr, frames_pil = read_frames(tmp_path, indices)

        clip_feats = extract_clip(frames_pil)          # [32, 512]
        diff_feats = extract_framediff(frames_bgr)     # [32, 49]
        obj_feats  = extract_objects(frames_pil)       # [32, 5, 324]

        os.makedirs(out_dir, exist_ok=True)
        torch.save(clip_feats, f'{out_dir}/clip.pt')
        torch.save(diff_feats, f'{out_dir}/framediff.pt')
        torch.save(obj_feats,  f'{out_dir}/objects.pt')
    except Exception as e:
        errors.append((vid, str(e)))
    finally:
        os.unlink(tmp_path)  # delete temp video immediately

print(f'Done. Errors: {len(errors)}')
if errors: print(errors[:5])

100%|██████████| 1614/1614 [3:49:43<00:00,  8.54s/it]

Done. Errors: 0


## Step 6 — Train Scene Classifier & Extract Scene Features

In [14]:
# Upload Places365 subset (data/places365_subset/) to Drive/ee508/data/places365_subset/
# OR just train it from scratch here using the CLIP features already extracted

import os
import torch
from src.scene_classifier import SceneClassifier, train_scene_classifier

SCENE_CKPT = f'{DRIVE_ROOT}/checkpoints/scene_mlp.pt'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

if not os.path.exists(SCENE_CKPT):
    # Notice we changed data_dir to features_dir here!
    scene_model = train_scene_classifier(
        features_dir=f'{DRIVE_ROOT}/data/places365_subset',
        device=device,
        epochs=20
    )

    torch.save(scene_model.state_dict(), SCENE_CKPT)
    print('Scene classifier saved.')
else:
    scene_model = SceneClassifier().to(device)
    scene_model.load_state_dict(torch.load(SCENE_CKPT, map_location=device))
    print('Scene classifier loaded.')

scene_model.eval()

Scene classifier saved.


SceneClassifier(
  (mlp): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=6, bias=True)
  )
)

In [ ]:
# Extract scene features from CLIP features already in Drive
scene_dirs = [d for d in os.listdir(FEATURES_DIR) if os.path.exists(f'{FEATURES_DIR}/{d}/clip.pt')]
print(f'Processing {len(scene_dirs)} videos for scene features...')

for vid in tqdm(scene_dirs):
    out_path = f'{FEATURES_DIR}/{vid}/scene.pt'
    if os.path.exists(out_path):
        continue
    clip_feats = torch.load(f'{FEATURES_DIR}/{vid}/clip.pt').to(device)  # [32, 512]
    with torch.no_grad():
        scene_logits = scene_model(clip_feats)  # [32, 6]
    torch.save(scene_logits.cpu(), out_path)

print('Scene features done.')

## Step 7 — Download Features to PC

After this notebook finishes, download `MyDrive/ee508/features/` to your local `ee508/features/` using `rclone` or the Colab file browser.

### Fastest: use rclone on your PC
```bash
# Install rclone (first time only)
curl https://rclone.org/install.sh | sudo bash
rclone config  # set up Google Drive remote called 'gdrive'

# Download features
rclone copy gdrive:ee508/features ~/studies/project/ee508/features/ --progress
rclone copy gdrive:ee508/data     ~/studies/project/ee508/data/     --progress
```

Then run training locally:
```bash
cd ~/studies/project/ee508
python3 scripts/prepare_labels.py
python3 scripts/split_dataset.py
python3 train.py
```